## Implement RAG with Lakebase Postgres

### Installing Utilities and Libraries

In [ ]:
%pip install psycopg[binary]==3.3.4 psycopg_pool==3.3.1 "databricks-sdk>=0.89.0" langchain-community==0.4.1 databricks-openai==0.17.1

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Setting up the Environment

In [ ]:
from databricks.sdk import WorkspaceClient
import psycopg

# Databricks SDK uses your existing OAuth identity
w = WorkspaceClient()

# Lakebase endpoint resource name
endpoint = (
    "projects/<project-id>/"
    "branches/<branch-id>/"
    "endpoints/<endpoint-id>"
)

# Generate a short-lived OAuth credential
credential = w.postgres.generate_database_credential(
    endpoint=endpoint
)

# Generates the workspace host URL
workspace_host = w.config.host.rstrip("/")

In [ ]:
host = "LAKEBASE_HOTSNAME"
db_name = "LAKEBASE_DATABASE_NAME"
username = "LAKEBASE_USERNAME"
password = credential.token

### Create a Connection Pool

In [ ]:
from psycopg_pool import ConnectionPool

pool = ConnectionPool(
    conninfo=(
        f"host={host} "
        f"dbname={db_name} "
        f"user={username} "
        f"password={password} "
        f"sslmode=require"
    ),
    min_size=2,
    max_size=10
)

pool.wait()

print("Connection pool created successfully")

### Creating the OpenAI Client

In [ ]:
from databricks_openai import DatabricksOpenAI

client = DatabricksOpenAI()

### Create the Embedding Generator Helper Function

In [ ]:
def generate_embeddings(text):

    # OpenAI Request
    completion = client.embeddings.create(
        model="databricks-gte-large-en",
        input=text
    )

    return completion.data[0].embedding

### Generate Vector Embeddings for the User Query

Some other questions to ask:

1) What is GreenSteel Ltd target year for achieving carbon neutrality?
2) By how much did FutureEnergy Corp increase its renewable generation capacity?
3) Compare the Scope 3 emissions challenges faced by GreenSteel and UrbanRetail. Also compare their company performance according to their ESG reports.
4) Which company has made the most progress on emission reductions, and why?

In [ ]:
user_query = "How is GreenSteel reducing emissions?"

query_embedding = generate_embeddings(user_query)

### Implement a Hybrid Search Query

In [ ]:
search_query = """
SELECT

    ChunkID,
    CompanyName,

    (
        (
            1 -
            (
                ChunkEmbedding <=> %s::vector
            )
        ) * 0.7

        +

        ts_rank(
            to_tsvector(
                'english',
                ChunkText
            ),
            plainto_tsquery(
                'english',
                %s
            )
        ) * 0.3

    ) AS hybrid_score,

    ChunkText

FROM RAG.ESG_Chunks

WHERE

    to_tsvector(
        'english',
        ChunkText
    )

    @@

    plainto_tsquery(
        'english',
        %s
    )

    OR

    (
        ChunkEmbedding <=> %s::vector
    ) < 0.5

ORDER BY hybrid_score DESC

LIMIT 10
"""

In [ ]:
from psycopg.rows import dict_row
import json

with pool.connection() as conn:

    with conn.cursor(
        row_factory=dict_row
    ) as cur:

        cur.execute(
            search_query,
            (
                query_embedding,  # ChunkEmbedding <=> %s::vector
                user_query,    # plainto_tsquery text
                user_query,    # plainto_tsquery text
                query_embedding   # ChunkEmbedding <=> %s::vector
            )
        )

        results = cur.fetchall()

rag_context = {
    "user_query": user_query,
    "documents": [
        {
            "chunk_id": result["chunkid"],
            "company_name": result["companyname"],
            "hybrid_score": float(result["hybrid_score"]),
            "content": result["chunktext"]
        }
        for result in results
    ]
}

print(json.dumps(rag_context, indent=4))

### Setting the System Prompt for the LLM Agent

In [ ]:
SYSTEM_PROMPT = """
You are an ESG and Sustainability Reporting Assistant for CarbonOps.

Your purpose is to answer user questions using only the information provided in the retrieved context. The retrieved context consists of ESG reports, sustainability disclosures, reviews, and company-specific ESG information.

Instructions:

1. Use only the information available in the provided context.
2. Do not make assumptions or invent facts.
3. If the answer cannot be determined from the context, clearly state:
   "I could not find sufficient information in the retrieved documents to answer that question."
4. When multiple companies are present in the retrieved context, clearly identify which company each statement refers to.
5. Summarize information in a concise, professional, and business-friendly manner.
6. When discussing ESG topics, pay attention to:
   - Carbon emissions
   - Scope 1, Scope 2, and Scope 3 emissions
   - Energy consumption
   - Renewable energy initiatives
   - Sustainability goals
   - ESG ratings and scores
   - Governance and compliance efforts
   - Risks and opportunities
7. If the user asks for a comparison, compare only the information present in the retrieved context.
8. If numerical values such as emissions, energy consumption, percentages, or ESG scores are present in the context, include them whenever relevant.
9. Do not mention embeddings, vector databases, chunking, retrieval pipelines, similarity scores, vector search, hybrid search, or any implementation details.
10. Maintain a professional consultant-style tone suitable for ESG analysts, sustainability managers, auditors, executives, and business stakeholders.

Response Guidelines:

- Answer the user's question directly.
- Use bullet points when presenting multiple findings.
- Highlight key ESG metrics when available.
- Provide a brief conclusion when appropriate.
- If the retrieved context contains conflicting information, explain the discrepancy.
- At the end of your response include a "Sources Used" section listing the companies referenced in the retrieved context.

The retrieved context provided in the conversation is the authoritative source of truth.
"""

### Sending API Call to LLM

In [ ]:
response = client.chat.completions.create(
    model="databricks-claude-sonnet-4-5",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
        Question:
        {user_query}

        Retrieved Context:
        {rag_context}
        """
                }
        ],
        temperature=0.2
)

print(response.choices[0].message.content)